# Phase 1 — lexical and semantic agreement studies

This notebook is an interactive, read-only view of the cached Phase 1 results. The
protocol and final decision record are `docs/phase1-design.md`; implementation and
regeneration commands are in `docs/phase1-implementation.md`. Running this notebook
performs no retrieval or model calls.

**Available grid.** RQ1 covers {DL19, DL20} × {top-10 all-pairs (*primary*),
uniform depth-100 (*rank-gap scope*)} × {Qwen 3.6 35B (primary, prompt v1),
Flan-T5-large (contrast, v0)}. The cached RQ2 WordNet analysis covers the two top-10
cells only; no uniform-condition RQ2 artefacts are present, so semantic conclusions are
bounded accordingly.
All axiom preferences below postdate the 2026-07-11 `ir_axioms` PROX fixes
(PROX2 orientation, PROX1 determinism), unlike the historical Phase 0 cache.

**Units and estimands.** The independent sampling unit for uncertainty is the query
(43 in DL19; 54 in DL20), not the pair. Coverage uses every sampled canonical pair.
Agreement uses only pairs on which the axiom is non-neutral and the order-collapsed
LLM verdict is decisive; its denominator therefore varies by axiom. Joint models use
decisive pairs and keep complete queries together in cross-validation folds. Top-10 is
the primary deployment-relevant condition; uniform depth-100 is a pre-specified
wide-gap scope condition. Qwen is primary and Flan-T5-large is a model contrast.

The sections follow the analysis chain: effectiveness reference → individual lexical
axioms → sampling control → joint fit → relaxed preconditions → semantic ablation.
Per-axiom comparisons are exploratory and are not adjusted for multiplicity.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from axiomrank.paths import results_dir

INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
PRIMARY = "models/qwen3.6-35B-A3B-AWQ"
CONTRAST = "google/flan-t5-large"
MODEL_COLOR = {PRIMARY: "#2a78d6", CONTRAST: "#1baf7a"}
MODEL_LABEL = {PRIMARY: "Qwen 3.6 35B", CONTRAST: "Flan-T5-large"}
COND_COLOR = {"top10": "#2a78d6", "uniform": "#4a3aa7"}
CELLS = ["dl19_top10", "dl19_uniform", "dl20_top10", "dl20_uniform"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": INK2,
    "axes.edgecolor": BASE, "font.size": 11,
})


def load_cells(experiment: str, filename: str) -> dict:
    """{(cell, model): DataFrame|dict} for every grid cell that has `filename`."""
    out = {}
    for path in sorted(results_dir(experiment).glob(f"dl*/metrics/*/{filename}")):
        cell, _, model_dir = path.relative_to(results_dir(experiment)).parts[:3]
        model = model_dir.replace("__", "/")
        if model not in MODEL_LABEL:  # ignore exploratory backends outside this study grid
            continue
        out[(cell, model)] = (
            pd.read_csv(path) if filename.endswith(".csv") else json.loads(path.read_text())
        )
    return out


rq1 = load_cells("rq1_lexical_agreement", "agreement.csv")
rq1_gap = load_cells("rq1_lexical_agreement", "gap_agreement.csv")
rq1_fit = load_cells("rq1_lexical_agreement", "joint_fit.json")
rq2 = load_cells("rq2_semantic_agreement", "agreement.csv")
rq2_fit = load_cells("rq2_semantic_agreement", "joint_fit.json")
eff = load_cells("ranking_effectiveness", "effectiveness.json")

required = {
    "RQ1": (rq1, {(c, m) for c in CELLS for m in (PRIMARY, CONTRAST)}),
    "RQ2": (rq2, {(f"dl{year}_top10", m) for year in (19, 20)
                  for m in (PRIMARY, CONTRAST)}),
    "effectiveness": (eff, {(f"dl{year}_top10", m) for year in (19, 20)
                    for m in (PRIMARY, CONTRAST)}),
}
missing = {name: sorted(keys - set(data)) for name, (data, keys) in required.items()
           if keys - set(data)}
if missing:
    raise FileNotFoundError(f"Incomplete Phase 1 metric grid: {missing}")

status = pd.DataFrame(
    {
        f"{label} · {MODEL_LABEL[m]}": {
            cell: "✓" if (cell, m) in data else "—" for cell in CELLS
        }
        for label, data in [("RQ1", rq1), ("RQ2", rq2), ("effectiveness", eff)]
        for m in [PRIMARY, CONTRAST]
    }
)
status  # effectiveness and RQ2 are defined only on the top-10 condition

## 1. Effectiveness reference — does pairwise aggregation improve BM25?

Cached top-10 verdicts are aggregated with Copeland scoring (PRP-allpair) and compared
with the original BM25 order using nDCG@10 and the same TREC qrels. The pre-specified
primary comparison concerns Qwen on both collections; Flan-T5-large is descriptive. The paired
bootstrap resamples query-level nDCG differences (10,000 draws, fixed seed). Its
percentile interval quantifies query variation; it is not a formal randomisation test.

In [ ]:
eff_perq = load_cells("ranking_effectiveness", "effectiveness.csv")

rng = np.random.default_rng(0)
rows = []
for (cell, model), e in sorted(eff.items()):
    m = e["metrics"]["nDCG@10"]
    d = eff_perq[(cell, model)]["nDCG@10_delta"].to_numpy()
    boot = rng.choice(d, size=(10_000, d.size), replace=True).mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    rows.append({
        "collection": cell.split("_")[0].upper(), "model": model,
        "bm25": m["mean_baseline"], "reranked": m["mean_reranked"],
        "delta": m["mean_delta"], "ci_lo": lo, "ci_hi": hi,
        "wtl": f'{m["wins"]}/{m["ties"]}/{m["losses"]}',
    })
effectiveness = pd.DataFrame(rows).sort_values(["collection", "model"], ascending=[True, False])

fig, ax = plt.subplots(figsize=(8, 2.9))
for y, r in enumerate(effectiveness.itertuples()):
    ax.plot([r.bm25, r.reranked], [y, y], color=GRID, linewidth=2, zorder=1)
    ax.scatter(r.bm25, y, s=60, color=MUTED, zorder=2)
    ax.scatter(r.reranked, y, s=72, color=MODEL_COLOR[r.model], zorder=3)
    ax.text(r.reranked + 0.006, y,
            f"+{r.delta:.3f} [{r.ci_lo:+.3f}, {r.ci_hi:+.3f}]   W/T/L {r.wtl}",
            va="center", fontsize=9, color=INK2)
ax.set_yticks(range(len(effectiveness)),
              [f"{r.collection} · {MODEL_LABEL[r.model]}" for r in effectiveness.itertuples()])
ax.invert_yaxis()
ax.set_xlim(0.45, 0.68)
ax.set_xlabel("nDCG@10  (grey = BM25 baseline, colored = Copeland-reranked; "
              "Δ with 95% paired query-bootstrap CI)")
ax.set_title("Depth-matched effectiveness: PRP-allpair vs BM25", loc="left", fontsize=12)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(length=0)
plt.tight_layout()
plt.show()

effectiveness.set_index(["collection", "model"]).round(3)

**Observed effectiveness.** On the judged-query sets, Qwen's Copeland reranking improves
mean nDCG@10 over BM25 by +0.069 (DL19) and +0.062 (DL20); the paired query-bootstrap
intervals exclude zero. This establishes that the cached primary verdicts contain
useful ranking signal under this aggregation protocol. It does **not** establish that
every disagreement with the classical axioms is relevant, causal, or transferable to
other collections. Flan-T5-large also improves the observed means, but was not part of
the primary comparison. Comparisons to unspecified literature anchors are omitted because
systems, candidate pools, prompts, and evaluation conventions differ.

## 2. RQ1 — per-axiom agreement profiles (Qwen)

The plotted estimand is conditional agreement, with query-bootstrap 95% percentile
intervals. Blue is top-10 all-pairs (primary); violet is the uniform depth-100 control.
TFC3 and M-TDC are omitted from this figure because they have fewer than 15 evaluable
pairs per cell, but remain visible in the artefacts and relaxation analysis. Grey marks
fewer than 30 evaluable pairs; this is a display convention, not a significance rule.

In [ ]:
MIN_N = 30
base = [a for a in rq1[("dl19_top10", PRIMARY)].axiom
        if "@" not in a and a not in ("TFC3", "M_TDC")]
order = (pd.concat([rq1[(c, PRIMARY)].set_index("axiom").agreement for c in CELLS
                    if (c, PRIMARY) in rq1], axis=1)
         .loc[base].mean(axis=1).sort_values().index.tolist())

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.6), sharey=True)
for ax, coll in zip(axes, ["dl19", "dl20"]):
    ax.axvline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
    for cond, dy in [("top10", 0.16), ("uniform", -0.16)]:
        key = (f"{coll}_{cond}", PRIMARY)
        if key not in rq1:
            continue
        df = rq1[key].set_index("axiom").loc[order]
        ok = df.n_evaluable >= MIN_N
        y = [order.index(a) + dy for a in df.index]
        ax.errorbar(df.agreement[ok], [v for v, k in zip(y, ok) if k],
                    xerr=[(df.agreement - df.ci_lo)[ok], (df.ci_hi - df.agreement)[ok]],
                    fmt="o", ms=6, color=COND_COLOR[cond], ecolor=GRID,
                    elinewidth=1.6, capsize=0, label=cond, zorder=3)
        ax.scatter(df.agreement[~ok], [v for v, k in zip(y, ok) if not k],
                   s=36, color=GRID, zorder=2)
    ax.set_title(coll.upper(), loc="left", fontsize=11, color=INK2)
    ax.set_xlim(0.28, 1.0)
    ax.set_xlabel("agreement with judge")
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.xaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
axes[0].set_yticks(range(len(order)), order)
axes[0].legend(loc="lower right", frameon=False, title=None)
fig.suptitle("RQ1 agreement profile, Qwen (grey = n < 30; bars = 95% bootstrap CI)",
             x=0.02, ha="left", fontsize=12)
plt.tight_layout()
plt.show()

The broad ordering is similar across collections: the largest observed agreements are
document-structure ones — **AND (~0.65–0.80) and LB1 (~0.73–0.77)** — not the classic
TF core. **TFC1 is near 0.5** in every cell (top-10 and uniform), and **DIV is at
or below chance** despite firing on > 90% of pairs. The proximity family lands in
between (~0.55–0.68). Note the CIs: with 43–54 queries most axioms' intervals span
±0.05–0.10, so small cross-cell differences and rankings should not be over-read.
Because axioms fire on different subsets, agreement magnitudes are not strictly
like-for-like comparisons and 0.5 is only the balanced binary reference.

## 3. Rank-gap scope

A diagnostic hypothesis was that conditional agreement would rise with absolute BM25
rank gap if classical axioms better describe widely separated candidates. The
figure uses Qwen's uniform cells; each x value identifies the stored rank-gap bin.
Decisive rate has a different denominator (all sampled pairs in the bin) from each
axiom's agreement (only evaluable pairs), so the lines must not be subtracted directly.

In [ ]:
series = [("AND", "#2a78d6"), ("TFC1", "#1baf7a"), ("DIV", "#eda100")]

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), sharey=True)
for ax, coll in zip(axes, ["dl19", "dl20"]):
    key = (f"{coll}_uniform", PRIMARY)
    if key not in rq1_gap:
        ax.set_axis_off()
        continue
    g = rq1_gap[key]
    bins = g.groupby("gap_bin")[["decisive_rate"]].first()
    ax.axhline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
    ax.plot(bins.index, bins.decisive_rate, color=MUTED, linewidth=1.6,
            linestyle=(0, (1, 2)), marker=".", label="judge decisive rate")
    for name, c in series:
        s = g[g.axiom == name].set_index("gap_bin").agreement
        ax.plot(s.index, s, color=c, linewidth=2, marker="o", ms=4, label=name)
        ax.annotate(name, (s.index[-1], s.iloc[-1]), xytext=(5, 0),
                    textcoords="offset points", color=c, fontsize=9.5, va="center")
    ax.set_title(coll.upper(), loc="left", fontsize=11, color=INK2)
    ax.set_xlabel("BM25 rank gap (decile upper edge)")
    ax.set_xlim(0, 112)
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
axes[0].set_ylabel("agreement / rate")
axes[0].legend(loc="upper left", frameon=False, fontsize=9)
fig.suptitle("Agreement vs BM25 rank gap (uniform condition, Qwen)",
             x=0.02, ha="left", fontsize=12)
plt.tight_layout()
plt.show()

The expected gradient is **not generally observed**. The judge's decisive rate rises
cleanly with gap (~0.55 → ~0.70–0.75), and TFC1 climbs only in the widest deciles
(~0.6–0.7 at gap ≈ 100). But there is no broad "classical axioms visibly work on
wide-gap pairs" regime: AND is high at *every* gap, and DIV drifts **below** chance as
the gap widens (0.55 → 0.33–0.36). This is descriptive evidence of conditional
anti-alignment, not proof that DIV causes the judge's choice. Bin-level pair counts and
query-bootstrap intervals are needed before interpreting local changes. The missing
general gradient narrows the battery's empirical scope but does not invalidate the
replicated top-10 baseline.

## 4. Joint prediction — how well does the battery predict held-out verdicts?

An L2-regularised logistic model predicts decisive verdicts from all axiom votes at
once under query-grouped five-fold cross-validation: strict core (10 classical axioms)
versus full battery (+ AND/DIV/LB1 and relaxed variants). Complete queries stay in one
fold. The baseline always predicts the global majority label. Accuracy reports
predictive fidelity only; it is not a mechanistic measure.

In [ ]:
rows = []
for cell in CELLS:
    if (cell, PRIMARY) not in rq1_fit:
        continue
    jf = rq1_fit[(cell, PRIMARY)]
    rows.append({
        "cell": cell,
        "base rate": jf["strict_core"]["base_rate"],
        "strict core": jf["strict_core"]["cv_accuracy"],
        "full battery": jf["full_battery"]["cv_accuracy"],
        "AUC (full)": jf["full_battery"]["cv_auc"],
        "n pairs": jf["strict_core"]["n_decisive_pairs"],
    })
fit = pd.DataFrame(rows).set_index("cell")

fig, ax = plt.subplots(figsize=(7.5, 2.9))
for y, (cell, r) in enumerate(fit.iterrows()):
    ax.plot([r["base rate"], r["full battery"]], [y, y], color=GRID, lw=2, zorder=1)
    ax.scatter(r["base rate"], y, s=60, color=MUTED, zorder=2, label="base rate" if y == 0 else None)
    ax.scatter(r["strict core"], y, s=64, color="#2a78d6", zorder=3, label="strict core" if y == 0 else None)
    ax.scatter(r["full battery"], y, s=64, color="#4a3aa7", zorder=3, label="full battery" if y == 0 else None)
ax.set_yticks(range(len(fit)), fit.index)
ax.invert_yaxis()
ax.set_xlabel("CV accuracy on decisive pairs")
ax.set_title("Joint fit vs majority-class baseline (Qwen)", loc="left", fontsize=12)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(length=0)
ax.legend(loc="lower right", frameon=False, fontsize=9)
plt.tight_layout()
fit.round(3)

The strict Phase 0 core is nearly uninformative on its own: +0–2 accuracy points
over the majority-class base rate (on DL20 top-10, nothing at all). The additions do
the work — the full battery reaches 0.57–0.64 CV accuracy (AUC 0.63–0.67), +4–9
points over base. Coefficients and single-axiom profiles suggest that AND/LB1/DIV carry
much of the predictive signal, but correlated features prevent a causal attribution.
This is RQ3's predictive starting point: roughly 57–64% out-of-fold accuracy on
decisive labels. Its out-of-fold errors seed Phase 2 diagnostics but do not quantify a
missing mechanism.

## 5. Relaxed preconditions — coverage vs agreement

Design §5.2: margin-relaxed variants of the strict-precondition axioms
(LNC1@tf{0.2,0.5}, TF-LNC@len{0.1,0.3}, M-TDC@mass{0.1,0.3}; TFC1/TFC3 relax the
shared LEN margin to {0.2,0.5}). The design's key question: does M-TDC's Phase 0
agreement (0.83–0.89 on a 15-pair niche) survive the coverage gain?

In [ ]:
FAMILIES = {
    "LNC1": ["LNC1", "LNC1@tf0.2", "LNC1@tf0.5"],
    "TF-LNC": ["TF_LNC", "TF_LNC@len0.1", "TF_LNC@len0.3"],
    "M-TDC": ["M_TDC", "M_TDC@mass0.1", "M_TDC@mass0.3"],
}

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.7), sharey=True)
for ax, (fam, members) in zip(axes, FAMILIES.items()):
    ax.axhline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)))
    for cell in CELLS:
        df = rq1[(cell, PRIMARY)].set_index("axiom").loc[members]
        cond = "top10" if cell.endswith("top10") else "uniform"
        ax.plot(df.coverage, df.agreement, marker="o", ms=5,
                color=COND_COLOR[cond],
                alpha=1.0 if cell.startswith("dl19") else 0.45,
                label=cell if fam == "LNC1" else None)
    df = rq1[("dl19_top10", PRIMARY)].set_index("axiom").loc[members]
    for name, cov, agr in zip(members, df.coverage, df.agreement):
        ax.annotate(name.split("@")[1] if "@" in name else "strict", (cov, agr),
                    xytext=(0, 9), textcoords="offset points", ha="center",
                    fontsize=8, color=INK2)
    ax.set_xscale("log")
    ax.set_title(fam, loc="left", fontsize=11, color=INK2)
    ax.set_xlabel("coverage (log)")
    ax.spines[["top", "right"]].set_visible(False)
    ax.yaxis.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(length=0)
axes[0].set_ylabel("agreement")
axes[0].legend(loc="lower left", frameon=False, fontsize=8.5)
fig.suptitle("Relaxing preconditions: coverage up, agreement down to chance (Qwen)",
             x=0.02, ha="left", fontsize=12)
plt.tight_layout()
plt.show()

rel = [a for members in FAMILIES.values() for a in members] + [
    "TFC1", "TFC1@len0.2", "TFC1@len0.5"]
pd.DataFrame({
    cell: rq1[(cell, PRIMARY)].set_index("axiom").loc[rel]
          .apply(lambda r: f"{r.agreement:.3f} ({int(r.n_evaluable)})", axis=1)
    for cell in CELLS
}).rename_axis("agreement (n evaluable)")

**In these cells, relaxation increases coverage without preserving the pilot's high
conditional agreement.** M-TDC's strict estimate of 0.83 (12 pairs, DL19) becomes
0.52–0.56 once the
single-difference precondition is dropped (274–855 pairs), and on DL20 even the strict
variant is near 0.5. The evidence is consistent with the Phase 0 estimate being unstable
and highly selected by the strict preconditions; it cannot identify an 'artefact' cause.
LNC1 decays toward and below chance as the TF margin widens; TF-LNC's relaxations sit at a weak
0.47–0.59. Two levers are degenerate: TFC1@len is bit-for-bit identical to
strict TFC1 (the LEN margin never binds on these pools), and TFC3 stays dead (≤ 4
evaluable pairs) at every margin, so this relaxation does not address its binding
precondition in these data. No relaxed variant earns a headline place on agreement;
they stay in
the feature set only for the joint fit above, where the regularised CV can still use
weak or anti signals.

## 6. RQ2 — do WordNet-tier semantic axioms add anything?

STMC1/STMC2/REG/ANTI-REG with the WordNet similarity backend, and the design §2.5
question: does adding them to the lexical battery improve the joint fit? Only top-10
artefacts exist for RQ2; the uniform wide-gap semantic control remains unmeasured.

In [ ]:
sem = pd.concat({cell: rq2[(cell, PRIMARY)].set_index("axiom").loc[
                     lambda d: d.index.str.endswith("@wn"),
                     ["coverage", "n_evaluable", "agreement", "ci_lo", "ci_hi"]]
                 for cell in CELLS if (cell, PRIMARY) in rq2}, names=["cell"])
display(sem.round(3))

delta = pd.DataFrame({
    cell: {
        "lexical cv_acc": jf["lexical"]["cv_accuracy"],
        "combined cv_acc": jf["combined"]["cv_accuracy"],
        "Δ accuracy": jf["combined"]["cv_accuracy"] - jf["lexical"]["cv_accuracy"],
        "Δ AUC": jf["combined"]["cv_auc"] - jf["lexical"]["cv_auc"],
    }
    for cell in CELLS if (cell, PRIMARY) in rq2_fit
    for jf in [rq2_fit[(cell, PRIMARY)]]
}).T
delta.round(3)

**Scoped semantic result and resource decision.** The per-axiom trigger nominally fires
twice on DL19 (STMC2@wn 0.555 at coverage 0.077; ANTI-REG@wn 0.556 at 0.619), but
neither repeats on DL20 (0.509 / 0.505), every semantic CI spans 0.5, and the
joint-fit comparison points the other way: adding the WordNet semantics *lowers* CV
accuracy and AUC in both primary-model top-10 cells (Δacc −0.7 to −1.6 points).
FastText was not run. This is evidence of **no detected incremental value for this
implementation and grid**, not evidence that semantic matching is generally useless;
WordNet coverage/crudeness and the lack of nested model-selection inference remain
limitations.

## Takeaways

1. **Effectiveness reference:** Qwen's aggregate reranking improves observed nDCG@10 over BM25 on
   both collections, supporting continued analysis of its top-10 preferences.
2. **The broad profile is robust on DL20** (profile correlation r ≈ 0.79–0.86 across
   collections, r = 0.93 across models on DL20): the classical TF core contributes
   little incremental predictive signal (TFC1 near 0.5); larger observed conditional
   agreements occur for
   AND (0.78–0.83) / LB1 (0.70–0.77) and, more weakly, proximity.
3. **The hypothesised gap gradient is unsupported:** judge decisiveness rises with gap,
   but per-axiom agreement mostly does not, and DIV moves below 0.5. This maps the
   battery's scope; no expected shape is required for pipeline validity.
4. **Relaxed preconditions increase coverage without stable agreement gains;** the
   12-pair Phase 0 M-TDC estimate does not survive broader applicability.
5. **No incremental WordNet benefit was detected** in the available top-10 RQ2 cells
   (combined < lexical for Qwen on DL19 and DL20). FastText was not run, and the uniform
   semantic condition remains unmeasured.
6. **Joint prediction:** the full battery reaches 0.57–0.64 CV accuracy (AUC 0.63–0.67) vs base
   rates 0.51–0.57. These are predictive-fidelity estimates only.

Phase 2 uses this corrected, non-degenerate lexical battery as its predictive baseline
and reports pooled results with query-set-specific checks; see `docs/phase1-design.md` §7.